In [2]:
import kagglehub


path = kagglehub.dataset_download("antareepdey/indian-railways")

print("Path to dataset files:", path)

100%|██████████| 1.50M/1.50M [00:01<00:00, 888kB/s]

Extracting files...


Path to dataset files: C:\Users\HP\.cache\kagglehub\datasets\antareepdey\indian-railways\versions\1


In [5]:
path=r"C:\Users\HP\IRCTC_cleaned.csv"
import pandas as pd
import os

df=pd.read_csv(path)
print(df)
import pandas as pd


print(df.columns.tolist())
print(df.dtypes)
print(df.head(5))




      train_no              train_name        source_station departure_time  \
0        10101    RN MADGAON EXP Train             Ratnagiri           1:40   
1        10102    MADGAON RN EXP Train               Madgaon          19:00   
2        10103         MANDOVI EXPRESS   C Shivaji Maharaj T           7:10   
3        10104         MANDOVI EXPRESS               Madgaon           9:15   
4        10105  DIVA SWV EXPRESS Train               Diva Jn           6:25   
...        ...                     ...                   ...            ...   
8361     69244      COR-ASV MEMU Train          Chittaurgarh           8:35   
8362     69245      ASV-HMT MEMU Train  Asarva Jn (Ahmedabad          19:20   
8363     69246      HMT-ASV MEMU Train           Himmatnagar           6:09   
8364     69249    SBIB-KTRD MEMU Train          Sabarmati Bg           6:30   
8365     69250    KTRD-SBIB MEMU Train          Katosan Road          19:05   

     arrival_time  distance   destination_station  

In [6]:
import pandas as pd
import sqlite3
import re
path=r"C:\Users\HP\IRCTC_cleaned.csv"
df = pd.read_csv(path)  # change filename to match yours

conn = sqlite3.connect("delays.db")

# ── TABLE 1: trains ───────────────────────────────────────────────────────────
# One row per train — clean metadata only
trains = df[[
    "train_no", "train_name", "source_station", "destination_station",
    "distance", "days_of_week", "classes"
]].copy()

trains.rename(columns={
    "train_no":            "train_number",
    "source_station":      "origin_station",
    "destination_station": "destination_station",
    "days_of_week":        "running_days",
})

trains.to_sql("trains", conn, if_exists="replace", index=False)
print(f"trains table: {len(trains)} rows")

# ── TABLE 2: schedule (exploded from intermediate_stops) ──────────────────────
# intermediate_stops looks something like:
# "NDLS|New Delhi|00:00|00:05|0, AGC|Agra Cantt|02:30|02:35|200, ..."
# The exact separator depends on the dataset — we'll detect it below.

def parse_stops(train_number, stops_text):
    """
    Parses the intermediate_stops blob into individual stop rows.
    Handles common formats: pipe-separated, comma-separated, newline-separated.
    Returns list of dicts.
    """
    if pd.isna(stops_text) or stops_text == "":
        return []

    rows = []

    # Try to split into individual stop chunks
    # Common formats seen in this dataset:
    # Format A: "CODE-Name-HH:MM-HH:MM-dist, CODE-Name-..."
    # Format B: "CODE|Name|HH:MM|HH:MM|dist\nCODE|..."
    # We'll try both

    # Normalise: try splitting by newline first, then comma
    if "\n" in stops_text:
        chunks = stops_text.strip().split("\n")
    else:
        # split on pattern: digit(s) followed by space and uppercase letters
        # this avoids splitting mid-name on commas
        chunks = re.split(r',\s*(?=[A-Z]{2,5}[\s\-\|])', stops_text)

    for seq, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if not chunk:
            continue

        # Try pipe separator first
        if "|" in chunk:
            parts = [p.strip() for p in chunk.split("|")]
        elif "-" in chunk:
            parts = [p.strip() for p in chunk.split("-", 4)]
        else:
            parts = chunk.split()

        if len(parts) < 2:
            continue

        rows.append({
            "train_number":   train_number,
            "stop_sequence":  seq,
            "raw_chunk":      chunk,          # keep raw for debugging
            "station_code":   parts[0] if len(parts) > 0 else None,
            "station_name":   parts[1] if len(parts) > 1 else None,
            "arrival_time":   parts[2] if len(parts) > 2 else None,
            "departure_time": parts[3] if len(parts) > 3 else None,
            "distance_km":    parts[4] if len(parts) > 4 else None,
        })

    return rows

# Print a sample of raw intermediate_stops so you can see the format
print("\nSample intermediate_stops values:")
for val in df["intermediate_stops"].dropna().head(3):
    print(repr(val[:300]))
    print("---")

# Explode all stops
all_stops = []
for _, row in df.iterrows():
    stops = parse_stops(row["train_no"], row["intermediate_stops"])
    all_stops.extend(stops)

schedule = pd.DataFrame(all_stops)
schedule.to_sql("schedule_raw", conn, if_exists="replace", index=False)
print(f"\nschedule_raw table: {len(schedule)} stop rows from {df['train_no'].nunique()} trains")
print(f"Avg stops per train: {len(schedule)/len(df):.1f}")

# ── TABLE 3: stations (unique stations extracted from schedule) ───────────────
stations = (
    schedule[["station_code", "station_name"]]
    .dropna()
    .drop_duplicates("station_code")
    .reset_index(drop=True)
)
stations["junction_rank"] = None  # fill this after analysis

stations.to_sql("stations", conn, if_exists="replace", index=False)
print(f"\nstations table: {len(stations)} unique stations")

# ── TABLE 4: running_days (normalised, one row per train per day) ─────────────
day_rows = []
for _, row in df.iterrows():
    if pd.isna(row["days_of_week"]):
        continue
    for day in str(row["days_of_week"]).split(","):
        day_rows.append({
            "train_number": row["train_no"],
            "day": day.strip().upper()
        })

running_days = pd.DataFrame(day_rows)
running_days.to_sql("running_days", conn, if_exists="replace", index=False)
print(f"running_days table: {len(running_days)} rows")

# ── TABLE 5: classes (normalised) ─────────────────────────────────────────────
class_rows = []
for _, row in df.iterrows():
    if pd.isna(row["classes"]):
        continue
    for cls in str(row["classes"]).split(","):
        class_rows.append({
            "train_number": row["train_no"],
            "class_code": cls.strip().upper()
        })

train_classes = pd.DataFrame(class_rows)
train_classes.to_sql("train_classes", conn, if_exists="replace", index=False)
print(f"train_classes table: {len(train_classes)} rows")

# ── JUNCTION RANK: which stations have the most trains? ───────────────────────
junction_counts = (
    schedule.groupby("station_code")["train_number"]
    .nunique()
    .reset_index()
    .rename(columns={"train_number": "train_count"})
    .sort_values("train_count", ascending=False)
)
junction_counts.to_sql("junction_rank", conn, if_exists="replace", index=False)

print("\nTop 15 junctions by train count:")
print(junction_counts.head(15).to_string(index=False))

conn.close()
print("\nAll tables written to delays.db")
print("Tables: trains, schedule_raw, stations, running_days, train_classes, junction_rank")

trains table: 8366 rows

Sample intermediate_stops values:
'Ratnagiri (arr=First,dep=01:40) | Rajapur Road (arr=02:41,dep=02:42) | Vaibhavwadi Road (arr=03:01,dep=03:02) | Kankavli (arr=03:43,dep=03:44) | Sindhudurg (arr=04:01,dep=04:02) | Kudal (arr=04:15,dep=04:16) | Zarap (arr=04:27,dep=04:28) | Sawantwadi Road (arr=04:41,dep=04:42) | Madure H (arr=04:53,'
---
'Madgaon (arr=First,dep=19:00) | Verna (arr=19:19,dep=19:20) | Karmali (arr=19:35,dep=19:36) | Thivim (arr=19:56,dep=19:57) | Pernem (arr=20:09,dep=20:10) | Madure H (arr=20:30,dep=20:31) | Sawantwadi Road (arr=20:42,dep=20:43) | Zarap (arr=20:57,dep=20:58) | Kudal (arr=21:19,dep=21:20) | Sindhudurg '
---
'C Shivaji Maharaj T (arr=First,dep=07:10) | Dadar (arr=07:22,dep=07:25) | Thane (arr=07:51,dep=07:55) | Panvel (arr=08:25,dep=08:30) | Mangaon (arr=10:14,dep=10:16) | Khed (arr=11:18,dep=11:20) | Chiplun (arr=11:46,dep=11:48) | Sangmeshwar (arr=12:22,dep=12:24) | Ratnagiri (arr=13:30,dep=13:35) | Ad'
---

schedule_raw table:

In [3]:
import pandas as pd
import sqlite3
import re
path= r"C:\Users\HP\IRCTC_cleaned.csv"
df = pd.read_csv(path)  # change to your filename

conn = sqlite3.connect("delays.db")

# ── TABLE 1: trains ───────────────────────────────────────────────────────────
trains = df[[
    "train_no", "train_name", "source_station", "destination_station",
    "distance", "days_of_week", "classes",
    "departure_time", "arrival_time"
]].copy()
trains.to_sql("trains", conn, if_exists="replace", index=False)
print(f"trains: {len(trains)} rows")

# ── TABLE 2: schedule — properly parsed ───────────────────────────────────────
# Format is: "Station Name (arr=HH:MM,dep=HH:MM) | Next Station (arr=...) | ..."
# "arr=First" means it's the origin (no arrival, only departure)
# "dep=Last"  means it's the destination (no departure, only arrival)

def parse_stops(train_no, train_name, stops_text):
    if pd.isna(stops_text) or not str(stops_text).strip():
        return []

    # Split on " | " — the consistent separator in this dataset
    chunks = [c.strip() for c in str(stops_text).split(" | ") if c.strip()]

    rows = []
    for seq, chunk in enumerate(chunks):
        # Extract: "Station Name (arr=HH:MM,dep=HH:MM)"
        match = re.match(
            r'^(.+?)\s*\(arr=([^,]+),dep=([^)]+)\)$',
            chunk.strip()
        )
        if not match:
            continue

        station_name = match.group(1).strip()
        arr_raw      = match.group(2).strip()
        dep_raw      = match.group(3).strip()

        # "First" = origin (first stop), "Last" = destination (last stop)
        arrival_time   = None if arr_raw == "First" else arr_raw
        departure_time = None if dep_raw == "Last"  else dep_raw
        is_origin      = arr_raw == "First"
        is_destination = dep_raw == "Last"

        rows.append({
            "train_no":        train_no,
            "train_name":      train_name,
            "stop_sequence":   seq,
            "station_name":    station_name,
            "arrival_time":    arrival_time,
            "departure_time":  departure_time,
            "is_origin":       is_origin,
            "is_destination":  is_destination,
        })

    return rows

print("Parsing intermediate_stops...")
all_stops = []
for _, row in df.iterrows():
    stops = parse_stops(row["train_no"], row["train_name"], row["intermediate_stops"])
    all_stops.extend(stops)

schedule = pd.DataFrame(all_stops)
schedule.to_sql("schedule", conn, if_exists="replace", index=False)
print(f"schedule: {len(schedule):,} stop rows from {df['train_no'].nunique():,} trains")
print(f"avg stops per train: {len(schedule)/len(df):.1f}")

# ── TABLE 3: stations — unique, with junction rank ────────────────────────────
junction_counts = (
    schedule.groupby("station_name")["train_no"]
    .nunique()
    .reset_index()
    .rename(columns={"train_no": "train_count"})
    .sort_values("train_count", ascending=False)
    .reset_index(drop=True)
)
junction_counts["junction_rank"] = junction_counts.index + 1
junction_counts.to_sql("stations", conn, if_exists="replace", index=False)
print(f"stations: {len(junction_counts):,} unique stations")

# ── TABLE 4: running_days (normalised) ────────────────────────────────────────
day_rows = []
for _, row in df.iterrows():
    if pd.isna(row["days_of_week"]):
        continue
    for day in str(row["days_of_week"]).split(","):
        day_rows.append({"train_no": row["train_no"], "day": day.strip().upper()})

running_days = pd.DataFrame(day_rows)
running_days.to_sql("running_days", conn, if_exists="replace", index=False)
print(f"running_days: {len(running_days):,} rows")

# ── TABLE 5: train_classes (normalised) ───────────────────────────────────────
class_rows = []
for _, row in df.iterrows():
    if pd.isna(row["classes"]):
        continue
    for cls in str(row["classes"]).split(","):
        class_rows.append({"train_no": row["train_no"], "class_code": cls.strip().upper()})

train_classes = pd.DataFrame(class_rows)
train_classes.to_sql("train_classes", conn, if_exists="replace", index=False)
print(f"train_classes: {len(train_classes):,} rows")

# ── VERIFY: show sample parsed stops for one train ────────────────────────────
sample_train = df["train_no"].iloc[2]
sample = schedule[schedule["train_no"] == sample_train]
print(f"\nSample — train {sample_train} ({df[df['train_no']==sample_train]['train_name'].iloc[0]}):")
print(sample[["stop_sequence","station_name","arrival_time","departure_time"]].to_string(index=False))

# ── TOP JUNCTIONS ─────────────────────────────────────────────────────────────
print("\nTop 20 junctions by train count:")
print(junction_counts.head(20)[["station_name","train_count","junction_rank"]].to_string(index=False))

con.close()
print("\nDone. Tables: trains, schedule, stations, running_days, train_classes")



ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject